# Growing Neural Cellular Automata — gecko baseline

Reproduction of [Mordvintsev et al., *Growing Neural Cellular Automata*, Distill 2020](https://distill.pub/2020/growing-ca/) in PyTorch.

**Runtime → Change runtime type → GPU** before running. The library code lives in `diffluoditty/`; this notebook just drives it.
Training is interruptible: stop the training cell at any time, poke around, and re-run it to continue.

In [ ]:
# Setup: clone the repo (Colab) or use the local checkout.
import os
if os.path.exists("../diffluoditty/__init__.py"):   # running from notebooks/ in a local checkout
    %cd ..
elif not os.path.exists("diffluoditty/__init__.py"):
    if not os.path.exists("diffluoditty"):
        !git clone -q https://github.com/aphid91/diffluoditty.git
    %cd diffluoditty
!git pull -q

# Re-import edited library code automatically, so you can hack on the .py files mid-session.
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML

from diffluoditty import NCA, Trainer, experiment_config, load_emoji, pad_target, circle_masks, half_mask
from diffluoditty import evaluate, load_runs, nca_param_count, shapes_for_budget
from diffluoditty import viz

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

## Target
The 🦎 emoji, shrunk to 40px and padded by 16px on each side → 72×72, with premultiplied alpha.
`load_image(path)` works for any RGBA PNG.

In [ ]:
target = pad_target(load_emoji("gecko", max_size=40), padding=16)
print(target.shape)
viz.show_images(viz.to_rgb(target)[None], scale=3)

## Architecture
Pick a preset, or `custom` to use the slider and text box. With one hidden layer of width H and C channels, the update MLP has **4·C·H + H + C** parameters (the perception filters are fixed, not learned).
C includes RGBA, so C=4 means no hidden-state channels at all: cells can only talk to each other through their colour.

In [ ]:
#@title Architecture { run: "auto" }
PRESET = "paper (8.3k)"  #@param ["paper (8.3k)", "~800", "~100", "custom"]
CHANNELS = 16  #@param {type:"slider", min:4, max:32, step:1}
HIDDEN = "128"  #@param {type:"string"}
#@markdown `HIDDEN`: one width (`128`), several layers (`16,16`), or `0` for a purely linear update.

PRESETS = {"paper (8.3k)": (16, "128"), "~800": (12, "16"), "~100": (6, "4")}
if PRESET != "custom":
    CHANNELS, HIDDEN = PRESETS[PRESET]
hidden = [int(h) for h in str(HIDDEN).split(",") if h.strip()]
n_params = nca_param_count(CHANNELS, hidden)

print(f"channels = {CHANNELS}  (RGBA + {CHANNELS - 4} hidden-state)")
print(f"hidden   = {hidden}")
print(f"=> {n_params:,} trainable parameters\n")
print(f"Other one-hidden-layer shapes with about {n_params:,} parameters (C, H, count):")
print("   " + "   ".join(f"({c}, {h}, {n})" for c, h, n in shapes_for_budget(n_params, tolerance=0.1)))

## Training settings
The defaults are the paper's. Each run gets a name built from these settings, and its checkpoint is saved under `checkpoints/` so you can compare runs at the end of the notebook.

| experiment | `use_pool` | `damage_n` | learns to… |
|---|---|---|---|
| `growing` | False | 0 | grow from a seed (but tends to explode or decay afterwards) |
| `persistent` | True | 0 | grow, and then *stay* that shape |
| `regenerating` | True | 3 | grow, persist, and repair damage |

Anything not shown here can still be overridden in code, e.g. `experiment_config(..., min_steps=48, grad_checkpoint_every=16)`.

In [ ]:
#@title Training { run: "auto" }
EXPERIMENT = "regenerating"  #@param ["growing", "persistent", "regenerating"]
N_ITERS = 8000  #@param {type:"integer"}
LR = 2e-3  #@param {type:"number"}
LR_DECAY_AT = 2000  #@param {type:"integer"}
#@markdown The learning rate is multiplied by 0.1 at iteration `LR_DECAY_AT`.
BATCH_SIZE = 8  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}

cfg = experiment_config(EXPERIMENT, lr=LR, lr_milestones=[LR_DECAY_AT], batch_size=BATCH_SIZE)
RUN_NAME = f"{EXPERIMENT}_c{CHANNELS}_h{'-'.join(map(str, hidden)) or '0'}_lr{LR:g}_s{SEED}"
print("run:", RUN_NAME)
cfg

Running this cell **starts a fresh model**. To train another architecture: change the forms above, then run from here down (*Runtime → Run after*).

In [ ]:
torch.manual_seed(SEED)
nca = NCA(size=tuple(target.shape[-2:]), channels=CHANNELS, hidden=hidden, fire_rate=0.5).to(device)
trainer = Trainer(nca, target, cfg)
print(f"{RUN_NAME}: {sum(p.numel() for p in nca.parameters()):,} parameters")

## Train
The plot shows log10 loss, and the last batch before and after its rollout. In `regenerating` mode the first sample in each batch is a fresh seed and the last three are damaged.
Interrupt whenever you like; re-running continues where it stopped.

In [ ]:
try:
    trainer.train(N_ITERS - trainer.step_count, callback=viz.show_progress, callback_every=100)
except KeyboardInterrupt:
    print(f"interrupted at step {trainer.step_count}")

## Evaluate and save
A fixed, seeded benchmark of 16 runs from the seed, so runs of different sizes can be compared directly:
loss during the trained window (steps 64–96), loss at step 1000, loss 200 steps after random damage at step 200, and the fraction of runs that died or exploded.

In [ ]:
result = evaluate(nca, target)
print(f"{RUN_NAME}  ({sum(p.numel() for p in nca.parameters()):,} params)")
for k in ["grow_loss", "persist_loss", "regen_loss", "died", "exploded"]:
    print(f"  {k:13s} {result[k]:.4f}")

os.makedirs("checkpoints", exist_ok=True)
trainer.save(f"checkpoints/{RUN_NAME}.pt", run_name=RUN_NAME, eval=result)
# To get a trained model back later: nca = NCA.from_checkpoint(f"checkpoints/{RUN_NAME}.pt", device)

## Training diagnostics

In [ ]:
h = trainer.history
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
viz.plot_history(h, axes[0])
axes[1].semilogy([r["grad_norm"] for r in h], ".", markersize=1, alpha=0.3); axes[1].set_title("raw gradient norm (before normalisation)")
axes[2].plot([r["lr"] for r in h]); axes[2].set_title("learning rate")
plt.show()

A random sample of the pool. After training, these should all look like (possibly slightly damaged) geckos.

In [ ]:
if trainer.pool is not None:
    idx = torch.randperm(cfg.pool_size)[:32]
    viz.show_images(viz.to_rgb(nca.render(trainer.pool.state[idx])), ncols=8)

## Growth from a single seed

In [ ]:
frames = viz.record(nca, nca.seed(1), n_steps=300, every=2)
HTML(viz.animate(frames, scale=3).to_jshtml())

## Regeneration
Grow for 200 steps, cut four copies in different ways, then watch them heal. (Only the `regenerating` model is expected to do this well.)

In [ ]:
h, w = target.shape[-2:]
cuts = torch.cat([
    half_mask(1, h, w, "right"),
    half_mask(1, h, w, "top"),
    circle_masks(2, h, w),
]).to(device)

frames = viz.record(nca, nca.seed(4), n_steps=500, every=2, events={200: lambda s: nca.damage(s, cuts)})
HTML(viz.animate(frames, scale=2).to_jshtml())

## Long-term stability
Does the pattern hold its shape far beyond the 96 steps it was trained on?

In [ ]:
with torch.no_grad():
    x, snaps, times = nca.seed(1), [], [100, 500, 1000, 2000, 5000]
    for t in range(1, times[-1] + 1):
        x = nca.step(x)
        if t in times:
            snaps.append(viz.to_rgb(nca.render(x))[0])
viz.show_images(snaps, titles=[f"t={t}" for t in times], scale=2.5)

## Compare runs
Everything in `checkpoints/` from this session, sorted by parameter count. Delete a file to drop it from the comparison.

In [ ]:
import pandas as pd

runs = load_runs("checkpoints/*.pt", target, device)
display(pd.DataFrame([
    {"run": r["name"], "params": r["n_params"], **{k: r["eval"][k] for k in ["grow_loss", "persist_loss", "regen_loss", "died", "exploded"]}}
    for r in runs
]).set_index("run").style.format(precision=4))
viz.compare_runs(runs)

Save an animation to the Colab session (Files panel on the left): `viz.animate(frames).save("regen.gif", writer="pillow")` or `.save("regen.mp4")`.